# Time-Sensitive Network Models

In [ ]:
import os
import glob
import bz2
import pandas as pd
import numpy as np
import textstat
import spacy
import re
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import random
import math
import json
import networkx as nx
import community as community_louvain 
import leidenalg                       
import igraph as ig
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr, zscore
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, chi2_contingency, pearsonr, kendalltau
import seaborn as sns

In [ ]:
input_dir = Path('cosineMatrices')                # dir with adjacency matrices
output_dir = Path('nearest3NeighboursAdjacency')  # where to save the results
output_dir.mkdir(exist_ok=True)

files = list(input_dir.glob('*.csv'))

for file in files:
    print(f"Processing: {file.name}")
    similarity_df = pd.read_csv(file, index_col=0)

    similarity_df.index = similarity_df.index.map(str)
    similarity_df.columns = similarity_df.columns.map(str)

    adjacency_df = pd.DataFrame(0.0, index=similarity_df.index, columns=similarity_df.columns)

    for text in similarity_df.index:
        try:
            pub_year = int(str(text)[:4])
        except ValueError:
            print(f"Skipping {text}: could not extract year.")
            continue

        valid_texts = [
            t for t in similarity_df.columns
            if t != text and str(t)[:4].isdigit() and int(str(t)[:4]) < pub_year
        ]

        similarities = similarity_df.loc[text].loc[valid_texts]

        if not similarities.empty:
            top_two = similarities.nlargest(3)
            for neighbour, sim_score in top_two.items():
                adjacency_df.loc[text, neighbour] = sim_score

    out_file = output_dir / f"adjacency_top3_{file.stem}.csv"
    adjacency_df.to_csv(out_file)

    # Save version with 0s replaced by NaN for visone
    out_file_na = output_dir / f"adjacency_top3_NA_{file.stem}.csv"
    adjacency_df.replace(0, pd.NA).to_csv(out_file_na)

    print(f"Saved adjacency matrices to:\n - {out_file}\n - {out_file_na}")

In [ ]:
adjacency_dir = 'nearest3NeighboursAdjacency/'           # dir with adjacency matrices
canonisation_dir = 'canonisationScores/'                 # dir with canonisation score files
output_dir = 'networkMetrics_time_sensitive_3nn/'        # where to save the results
os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(adjacency_dir):
    if filename.endswith('.csv') and filename.startswith('adjacency_top3_features'):
        lang_code = filename.split('_')[-2].replace('.csv', '')
        print(lang_code)
        adj_path = os.path.join(adjacency_dir, filename)

        canon_filename = f'titles_with_scores_{lang_code}.csv'
        canon_path = os.path.join(canonisation_dir, canon_filename)

        if not os.path.exists(canon_path):
            print(f"Skipping {lang_code}: no canonisation file found at {canon_path}")
            continue

        sim_df = pd.read_csv(adj_path, index_col=0, encoding='utf8').fillna(0)

        canon_df = pd.read_csv(canon_path)
        if 'ID' not in canon_df.columns or 'canonisation_score' not in canon_df.columns:
            print(f"Skipping {lang_code}: canonisation file columns are not as expected.")
            continue

        G = nx.from_pandas_adjacency(sim_df, create_using=nx.DiGraph)

        # Compute centrality metrics
        indegree = dict(G.in_degree(weight='weight'))
        pagerank = nx.pagerank(G, weight='weight')
        betweenness = nx.betweenness_centrality(G, weight='weight', normalized=True)
        closeness = nx.closeness_centrality(G, distance=lambda u, v, d: 1/d.get('weight', 1) if d.get('weight', 1) != 0 else float('inf'))
        
        # Louvain clustering 
        G_undirected = G.to_undirected()
        louvain_partition = community_louvain.best_partition(G_undirected, weight='weight')

        # Leiden clustering
        g_igraph = ig.Graph.from_networkx(G_undirected)
        leiden_partition = leidenalg.find_partition(g_igraph, leidenalg.RBConfigurationVertexPartition, weights=g_igraph.es['weight'])

        node_list = list(G.nodes)
        canon_series = canon_df.set_index('ID').reindex(node_list)['canonisation_score']

        results = pd.DataFrame({
            "title": node_list,
            "indegree": [indegree.get(node, 0) for node in node_list],
            "pagerank": [pagerank.get(node, 0.0) for node in node_list],
            "betweenness": [betweenness.get(node, 0.0) for node in node_list],
            "closeness": [closeness.get(node, 0.0) for node in node_list],
            "canonisation_score": canon_series.values,
            "louvain_cluster": [louvain_partition.get(node, -1) for node in node_list],
            "leiden_cluster": [leiden_partition.membership[i] for i in range(len(node_list))],
        })

        out_path = os.path.join(output_dir, f'network_metrics_{lang_code}.csv')
        results.to_csv(out_path, index=False)
        print(f"Saved metrics and clusters for {lang_code} to {out_path}")

In [ ]:
input_dir = "networkMetrics_time_sensitive_3nn"
output_dir = "networkMetrics_time_sensitive_3nn/temporalTrend"
os.makedirs(output_dir, exist_ok=True)

# Grouping parameter: e.g. 10 for decade bins
BIN_SIZE = 10

for filename in os.listdir(input_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(input_dir, filename)
        df = pd.read_csv(filepath)

        df["pub_year"] = df["title"].str.replace(r"(\d{4}).*", r"\1", regex=True)
        df["pub_year"] = pd.to_numeric(df["pub_year"], errors="coerce")

        if "canonisation_score" not in df.columns:
            print(f"Skipping {filename}: missing canonisation_score.")
            continue

        # Bin years (e.g., 1890s, 1900s)
        df["year_bin"] = (df["pub_year"] // BIN_SIZE) * BIN_SIZE

        grouped = df.groupby("year_bin")["canonisation_score"].mean().reset_index()

        plt.figure(figsize=(6, 4))
        plt.plot(grouped["year_bin"], grouped["canonisation_score"], marker="o", linestyle="-")
        plt.title(f"Canonisation Trend – {filename.replace('.csv', '')}")
        plt.xlabel("Year of Publication (binned)")
        plt.ylabel("Average Canonisation Score")
        plt.grid(True)
        plt.tight_layout()

        plot_name = filename.replace(".csv", ".png")
        plt.savefig(os.path.join(output_dir, plot_name))
        plt.close()

In [ ]:
metrics_dir = 'networkMetrics_time_sensitive_3nn/'
output_dir = 'clusterCorrelationResults_time_sensitive_3nn/'
os.makedirs(output_dir, exist_ok=True)

results = [] 

for filename in os.listdir(metrics_dir):
    if filename.endswith('.csv') and filename.startswith('network_metrics'):
        lang_code = filename.replace('network_metrics_', '').replace('.csv', '')
        file_path = os.path.join(metrics_dir, filename)

        print(f"Processing {lang_code}...")

        df = pd.read_csv(file_path)

        df_clean = df.dropna(subset=[
            'canonisation_score', 'indegree', 'pagerank', 'betweenness', 'closeness'
        ])

        df_clean['log_indegree'] = np.log1p(df_clean['indegree'])

        # Compute Spearman correlations
        rho_indegree, p_indegree = spearmanr(df_clean['canonisation_score'], df_clean['log_indegree'])
        rho_pagerank, p_pagerank = spearmanr(df_clean['canonisation_score'], df_clean['pagerank'])
        rho_betweenness, p_betweenness = spearmanr(df_clean['canonisation_score'], df_clean['betweenness'])
        rho_closeness, p_closeness = spearmanr(df_clean['canonisation_score'], df_clean['closeness'])

        print(f"Results for {lang_code}:")
        print(f"Indegree (log): rho = {rho_indegree:.3f}, p = {p_indegree:.3e}")
        print(f"Pagerank:       rho = {rho_pagerank:.3f}, p = {p_pagerank:.3e}")
        print(f"Betweenness:    rho = {rho_betweenness:.3f}, p = {p_betweenness:.3e}")
        print(f"Closeness:      rho = {rho_closeness:.3f}, p = {p_closeness:.3e}")
        print()

        results.append({
            'language': lang_code,
            'rho_log_indegree': rho_indegree,
            'p_log_indegree': p_indegree,
            'rho_pagerank': rho_pagerank,
            'p_pagerank': p_pagerank,
            'rho_betweenness': rho_betweenness,
            'p_betweenness': p_betweenness,
            'rho_closeness': rho_closeness,
            'p_closeness': p_closeness
        })

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(output_dir, 'centrality_correlation_summary.csv'), index=False)
print(f"\nSaved summary to {output_dir}centrality_correlation_summary.csv")

In [ ]:
metrics_dir = 'networkMetrics_time_sensitive_3nn/'            
output_dir = 'clusterCorrelationResults_time_sensitive_3nn/'  
os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(metrics_dir):
    if filename.endswith('.csv') and filename.startswith('network_metrics_'):
        lang_code = filename.replace('network_metrics_', '').replace('.csv', '')
        file_path = os.path.join(metrics_dir, filename)

        print(f"Processing {lang_code}...")

        df = pd.read_csv(file_path)

        required_cols = {'canonisation_score', 'indegree', 'pagerank', 'betweenness', 'closeness'}
        if not required_cols.issubset(df.columns):
            print(f"Skipping {lang_code}: missing required columns.")
            continue
        
        cluster_cols = [col for col in ['louvain_cluster', 'leiden_cluster'] if col in df.columns]
        if not cluster_cols:
            print(f"Skipping {lang_code}: no clustering columns found.")
            continue

        all_results = {col: [] for col in cluster_cols}

        for cluster_col in cluster_cols:
            for cluster_id, group in df.groupby(cluster_col):
                group_clean = group.dropna(subset=[
                    'canonisation_score', 'indegree', 'pagerank', 'betweenness', 'closeness'
                ])
                n = len(group_clean)

                if n > 3:
                    rho_indegree, p_indegree = spearmanr(group_clean['canonisation_score'], group_clean['indegree'])
                    rho_pagerank, p_pagerank = spearmanr(group_clean['canonisation_score'], group_clean['pagerank'])
                    rho_betweenness, p_betweenness = spearmanr(group_clean['canonisation_score'], group_clean['betweenness'])
                    rho_closeness, p_closeness = spearmanr(group_clean['canonisation_score'], group_clean['closeness'])
                else:
                    rho_indegree = p_indegree = np.nan
                    rho_pagerank = p_pagerank = np.nan
                    rho_betweenness = p_betweenness = np.nan
                    rho_closeness = p_closeness = np.nan

                all_results[cluster_col].append({
                    'language': lang_code,
                    'cluster_id': cluster_id,
                    'n': n,
                    'spearman_indegree': rho_indegree,
                    'p_indegree': p_indegree,
                    'spearman_pagerank': rho_pagerank,
                    'p_pagerank': p_pagerank,
                    'spearman_betweenness': rho_betweenness,
                    'p_betweenness': p_betweenness,
                    'spearman_closeness': rho_closeness,
                    'p_closeness': p_closeness,
                })

        for cluster_col, results in all_results.items():
            corr_df = pd.DataFrame(results)
            method_name = cluster_col.replace('_cluster', '')
            out_path = os.path.join(output_dir, f'{method_name}_cluster_correlation_{lang_code}.csv')
            corr_df.to_csv(out_path, index=False)
            print(f"Saved {method_name} cluster correlations for {lang_code} to {out_path}")
